# Ders 18: Kriptografik Makbuzlarla Yapay Zeka Ajanlarını Güvence Altına Alma

## Uygulamalı Defter

Bu defter dört egzersizi adım adım anlatır:

1. Bir ajan araç çağrısı için **ilk makbuzunuzu imzalayın** ve doğrulayın.
2. Makbuzda **değişiklik yapın** ve doğrulamanın başarısız olduğunu görün.
3. Üç makbuzdan oluşan bir zincir **oluşturun** ve zincir bütünlüğünü doğrulayın.
4. Her işlemin bir makbuz yaydığı bir şekilde Microsoft Agent Framework araç çağrısını **sarın**.

Tüm kriptografik ilkel işlemler iyi bakılan kütüphanelerden ithal edilmiştir (`pynacl` Ed25519 için, `jcs` RFC 8785 kanonik JSON için, `hashlib` Python standart kütüphanesinden SHA-256 için). Makbuz mantığı ise okunabilir ve değiştirilebilir düz Python kodudur.

Hücreleri sırasıyla çalıştırın. Her bölüm kısa ve kendi içinde bağımsızdır.


## Kurulum

İki bağımlılığı yükleyin. Her ikisi de izin verici lisanslara sahiptir (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Yardımcı Araçlar

Bu iki yardımcı, herhangi bir nesnenin base64url kodlamasını (dolgusuz) ve SHA-256 karmasını gerçekleştirir. Geri kalan not defteri, makbuz mantığına odaklanmaya devam eder.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Bölüm 1: İlk makbuzunu imzala

Düşünün ki **Contoso Travel** için çalışan temsilcimiz, bir müşteri için Sydney'den Los Angeles'a uçuşlara baktı. Gelecekte bir denetçinin, ayrı olarak sabitlenmiş bir yayıncı anahtarı altında doğrulayabilmesi için aracının bu araç çağrısına ilişkin beyanını kaydetmek istiyoruz.

### Adım 1.1: İmza anahtarı oluştur

Üretimde, temsilcinin imza anahtarı bir donanım güvenlik modülünde (HSM), Azure Key Vault'ta veya benzer korumalı bir depoda saklanır. Bu ders için bellekte yeni bir anahtar oluşturuyoruz.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Adım 1.2: Makbuz yükünü oluştur

Yük, makbuzun doğrulamasını istediğimiz her şeyi içerir: kim işlem yaptı, hangi araç kullanıldı, hangi argümanlarla, ne döndü, hangi politika altında ve ne zaman. Argümanları ve sonucu, makbuzun hassas içeriği sızdırmaması için doğrudan dahil etmek yerine özetleriz.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Adım 1.3: Makbuzu imzalama ve birleştirme

Üç adım:

1. Aynı mantıksal makbuzu üreten iki uygulamanın aynı baytları oluşturması için JCS kullanarak yükü kanonikleştirin.
2. Kanonik baytları doğrudan Ed25519 özel anahtarıyla imzalayın. PureEdDSA mesajı dahili olarak karmalar, bu yüzden ekstra bir ön-karma protokolü değiştirir.

İmza daha sonra nihai makbuzu üretmek için orijinal yüke eklenir.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Adım 1.4: Makbuzu doğrulayın

Doğrulama süreci tersine çevirir. İmzayı çıkarır, kanonik baytları yeniden hesaplar ve imzayı düzenleyicinin genel anahtarına karşı kontrol ederiz.

Doğrulama çevrimdışı yapılır: çağrılacak servis veya ağ erişimi yoktur. Yine de makbuz dışında bir şeye ihtiyaç duyar: Denetçinin zaten güvendiği bir kanal aracılığıyla elde edilen düzenleyicinin genel anahtarı, örneğin düzenleyicinin yayınladığı bir JWK Set veya önceden sabitlenmiş bir değer. Makbuz içindeki `public_key` alanı bu rolü oynayamaz, çünkü makbuzu yazan kişi aynı zamanda o alanı da yazmıştır. Bölüm 2, bunun önlediği sahteciliği gösterir.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


`Receipt is valid: True`, `Pre-hashed receipt valid: False` ve `Malformed receipts rejected: True` görmelisiniz. Pozitif durum, doğrudan-JCS imzalama yolunun çalıştığını kanıtlar; negatif kontrol, imza kapsamı kuralını yalnızca metin olarak bırakmak yerine yürütülebilir yapar; son satır, istisna yükseltmek yerine hatalı girdinin `False` döndürdüğünü gösterir.


## Bölüm 2: Makbuzu Değiştirme

Makbuzların tüm amacı, müdahaleye karşı dayanıklı olmalarıdır. Bunu kanıtlayalım.

Makbuzdaki tam olarak bir karakteri değiştireceğiz ve doğrulamanın başarısız olduğunu gözlemleyeceğiz.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### What just happened?

When we changed `policy_id`, the canonical bytes changed. The signature (which was over the original canonical bytes) no longer matches. Verification correctly returns `False`.

There is no way to change a signed field, meaning anything outside the `signature` object, and still have the receipt verify against the issuer's public key, unless the attacker has the issuer's private key. The `signature` object itself is not signed: `sig` is the signature, and `verify_receipt` ignores `public_key`, so editing that field alone changes nothing. As long as the private key is in a key vault and verifiers use the issuer's independently published key, a change to a signed field is impossible to hide. That condition matters: the next cell shows a modified receipt that verifies perfectly against a different key.

Try it yourself: modify the `tool_name` or `agent_id` or `timestamp` in the cell above and re-run. Every change to a signed field produces an invalid receipt.


### İmzanın yakalayamadığı tahrifat

Yukarıdaki saldırgan yeniden imzalamadı. Daha dikkatli biri yapardı: makbuzu değiştirir,
kendisinin oluşturduğu bir anahtarla imzalar ve o anahtarı
`public_key` alanına koyardı. Sonuç, imzası içeriğiyle tamamen gerçek bir makbuz olur.


Bunu çalıştırın ve hangi iki kontrolden birinin yakaladığını gözlemleyin.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


`True` sonra `False` görmelisiniz. Üç tane `assert` satırı bunu
bir kontrol yapar: sahtecilik, sahteciliğin düzenleyicinin anahtarına karşı doğrulanması halinde hata verir.


Sahtecilik dahili olarak tutarlıdır. İmzası gerçekten içeriği ve taşıdığı anahtarla eşleşir, bu yüzden
anahtarı makbuzdan okuyan bir doğrulayıcı bunu geçerli olarak rapor eder. Makbuzda yanlış görünen hiçbir şey yoktur, çünkü
hiçbir şey tutarsız değildir.


Tek yakalayan şey, saldırganın etkileme şansı olmayan bir anahtara karşı kontrol edilmesidir.
Bu nedenle `verify_receipt` anahtarı argüman olarak alır: 
`verify_receipt(receipt)` imzasına sahip bir fonksiyon güvenli bir şekilde yazılamaz,
çünkü ulaşabileceği tek anahtar saldırganın sağladığı anahtardır.

Aynı makbuz outside notebook incelemek isterseniz `sample_receipts/04_resigned_receipt.json` olarak kaydedilir.
defterin dışında incelemek istiyorum.


## Bölüm 3: Makbuzları zincir halinde birbirine bağlayın

Tek bir makbuz bir eylemi korur. Çoğu ajan birçok eylem gerçekleştirir. Tüm diziyi tahrifata karşı açık hale getirmek için, yeni makbuzun yüküne önceki makbuzun karmasını ekleyerek her makbuzu bir öncekine bağlarız.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Birisi sunulan dizi içindeki bir makbuzu kaldırır veya sırasını değiştirirse sonraki bağlantı kopar. Kısaltma veya imzalayan tarafından kontrol edilen eklerin yeniden yazılmasını tespit etmek için ayrıca sabitlenmiş mevcut baş, dış kontrol noktası ya da doğrulanmış nüfus taahhüdü gerekir.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Şimdi zinciri, ortadaki makbuzu tahrif ederek kırın ve yeniden doğrulayın. Tahrif edilmiş makbuz imza kontrolünden geçemiyor ve sonraki makbuz zincir bağlantısı kontrolünden geçemiyor (çünkü `previous_receipt_hash` artık değiştirilmiş ortadaki makbuzun hash'iyle eşleşmiyor).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Makbuz 0 hâlâ doğrulanıyor (değiştirilmedi ve bağlı olduğu bir önceki yok). Makbuz 1, `tool_args_hash` değiştirdiğimiz için imza kontrolünde başarısız oluyor. Makbuz 2, zincir bağlantı kontrolünde başarısız oluyor çünkü `previous_receipt_hash`, orijinal (şimdi değiştirilmiş) makbuz 1'e göre hesaplanmıştı.

Bir saldırgan değiştirilmiş makbuz 1'i yeniden imzalamış olsa bile, yeni imza yayıcının özel anahtarına sahip olmadıkça yayıcının anahtarına karşı başarısız olur (bölüm 2’nin gösterdiği gibi kendi anahtarlarıyla yeniden imzalama başarısız olur) ve makbuz 2’deki zincir bağlantı uyumsuzluğu yine değişikliği açığa çıkarır. Değişikliği gizlemek için, saldırgan değiştirme noktasından itibaren her makbuzu yeniden imzalamak zorunda kalır ki bu da yayıcının özel anahtarına sahip olmayı gerektirir.


## Bölüm 4: Bir ajan aracı çağrısını makbuz imzalamayla sarmalama

Gerçek bir dağıtımda, her ajan yazarının `make_receipt` çağırmayı hatırlamasını istemezsiniz. Her araç çağrısı için makbuz imzalamanın otomatik olmasını istersiniz.

İşte en basit desen: herhangi bir çağrılabilir araç işlevini alan ve bunun makbuz yayan bir sürümünü döndüren bir sarmalayıcı sınıf. Bu, Microsoft Agent Framework (`agent_framework.foundry`) dahil olmak üzere herhangi bir ajan çerçevesine uyum sağlar.

Eğer kurulu bir Microsoft Foundry projeniz yoksa, aşağıdaki yerel taklit yine de deseni gösterir.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Microsoft Agent Framework ile Entegrasyon

Yukarıdaki `ReceiptedTool` sarmalayıcı çerçeveden bağımsızdır. Microsoft Agent Framework ile oluşturulan bir ajan içinde kullanmak için, sarılı fonksiyonu bir araç olarak kaydedin. Bir taslak (gerçek Microsoft Foundry araç kaydı ile taklit değiştirilecektir):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Ajan çerçevesinin makbuzlar hakkında herhangi bir bilgi sahibi olması gerekmez. Makbuz imzalama araç etrafında sarmalanır, çerçeveye bağlanmaz. Bu, ajanı yeniden yazmadan mevcut ajan koduna kaynağı nasıl eklersiniz.


## Özet ve ekstra zorluk

Şunları yaptınız:

- Bir Ed25519 anahtar çifti oluşturuldu.
- Bir ajan aracı çağrısı için fiş oluşturuldu ve imzalandı.
- Fiş, yalnızca fişten bağımsız olarak elde edilen yayıncının açık anahtarı kullanılarak çevrimdışı doğrulandı.
- Bir fiş üzerinde oynandı ve doğrulamanın başarısız olduğunu gözlemlediniz.
- Üç fişlik zincirlenmiş bir karma dizisi oluşturuldu.
- Zincirin ortasında oynama yapıldı ve hem imza hem de zincir bağlantısı hatası gözlemlendi.
- Bir araç işlevi otomatik fiş imzalama ile sarıldı.

**Ekstra zorluk.** Fiş şemasına bir `request_id` alanı (dağıtılmış izleme için bir UUID) ekleyin. `make_receipt` öğesini bunu içerecek şekilde güncelleyin ve fişlerin uçtan uca doğrulandığını teyit edin. Sonra imzalamadan sonra alanı değiştirin ve doğrulamanın başarısız olduğunu teyit edin. Bu, her baytın kanonik kodlamada imzaya nasıl katkıda bulunduğunu içselleştirmenizi sağlar.

**Önemli sınır.** Sabitlenmiş bir anahtar ve zincir bağlamı altında, fişler imza atamasını, içerik bütünlüğünü ve sunulan önceki elemana bağlantıyı kanıtlar. Yürütmeyi, kaynak gerçeğini, güvenilir zamanı, bütünlüğü, politika değerlendirmesini veya doğruluğu kanıtlamazlar. Fişler bir temeldir. Yönetişim üzerine inşa ettiğiniz sistemdir.

Bu sınırı aklınızda tutarak dersi README dosyasını tekrar okuyun. Takımların fişlerle ilgili en yaygın hatası "fişlerimiz var" demenin "yönetiliyoruz" anlamına geldiğini varsaymaktır. Böyle değildir. Fişler ajan davranışının denetlenebilir olmasını sağlar. Doğru olmasını sağlamaz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
